<a href="https://colab.research.google.com/github/njalagam23/neural-networks-from-scratch/blob/main/day12_api_logging_error_handling_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 12 — Production API Reliability

## Objective
Improve our PyTorch customer churn prediction API using:
- Application logging
- Error handling
- Health monitoring
- Automated testing

## Business Scenario
A telecommunications company needs a reliable AI prediction
service that engineers can monitor and troubleshoot.

## Technologies
Python, PyTorch, FastAPI, Uvicorn, Pytest



In [1]:
from pathlib import Path
import subprocess

repo = Path("/content/churn-repo")

if not repo.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/njalagam23/neural-networks-from-scratch.git",
            str(repo)
        ],
        check=True
    )

project = repo / "customer-churn-app"

print("Project exists:", project.exists())
print("API exists:", (project / "api.py").exists())
print("Requirements exists:", (project / "requirements.txt").exists())
print("Tests folder exists:", (project / "tests").exists())

Project exists: True
API exists: False
Requirements exists: True
Tests folder exists: True


In [2]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")

api_file = project / "app" / "api.py"

print("API exists:", api_file.exists())
print("API location:", api_file)

if api_file.exists():
    print("\nAPI contents:")
    print(api_file.read_text())

API exists: True
API location: /content/churn-repo/customer-churn-app/app/api.py

API contents:

from fastapi import FastAPI
from pydantic import BaseModel, Field
from app.predict import ChurnPredictor

app = FastAPI(
    title="Customer Churn Prediction API",
    version="1.0.0"
)

predictor = ChurnPredictor()


class CustomerInput(BaseModel):
    usage: float = Field(ge=0)
    tenure: int = Field(ge=0)
    tickets: int = Field(ge=0)


@app.get("/")
def home():
    return {"message": "Churn Prediction API is running"}


@app.post("/predict")
def predict_churn(customer: CustomerInput):
    return predictor.predict(
        usage=customer.usage,
        tenure=customer.tenure,
        tickets=customer.tickets
    )



In [3]:
from pathlib import Path

api_file = Path(
    "/content/churn-repo/customer-churn-app/app/api.py"
)

print(api_file.read_text())


from fastapi import FastAPI
from pydantic import BaseModel, Field
from app.predict import ChurnPredictor

app = FastAPI(
    title="Customer Churn Prediction API",
    version="1.0.0"
)

predictor = ChurnPredictor()


class CustomerInput(BaseModel):
    usage: float = Field(ge=0)
    tenure: int = Field(ge=0)
    tickets: int = Field(ge=0)


@app.get("/")
def home():
    return {"message": "Churn Prediction API is running"}


@app.post("/predict")
def predict_churn(customer: CustomerInput):
    return predictor.predict(
        usage=customer.usage,
        tenure=customer.tenure,
        tickets=customer.tickets
    )



In [4]:
import logging
import time
from fastapi import Request

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("churn_api")

In [7]:
from pathlib import Path

api_file = Path("/content/churn-repo/customer-churn-app/app/api.py")

code = api_file.read_text()

# Add imports
if "import logging" not in code:
    code = "import logging\nimport time\nfrom fastapi import Request\n" + code

# Add logging configuration and middleware
middleware = '''
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("churn_api")


@app.middleware("http")
async def log_requests(request: Request, call_next):
    start_time = time.perf_counter()
    status_code = 500

    try:
        response = await call_next(request)
        status_code = response.status_code
        return response
    finally:
        duration_ms = (time.perf_counter() - start_time) * 1000

        logger.info(
            "method=%s path=%s status=%s latency_ms=%.2f",
            request.method,
            request.url.path,
            status_code,
            duration_ms
        )
'''

if '@app.middleware("http")' not in code:
    position = code.index("predictor = ChurnPredictor()")
    code = code[:position] + middleware + "\n" + code[position:]

api_file.write_text(code)

print("API updated successfully!")

API updated successfully!


In [8]:
code = api_file.read_text()

print("Logging imported:", "import logging" in code)
print("Middleware added:", '@app.middleware("http")' in code)
print("Prediction endpoint preserved:", '@app.post("/predict")' in code)

Logging imported: True
Middleware added: True
Prediction endpoint preserved: True


In [9]:
import subprocess
import sys
import time
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")
log_file = project / "uvicorn_day12.log"

# Stop the previous server if this notebook started one
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=10)

# Start the updated API
with open(log_file, "w") as log:
    server = subprocess.Popen(
        [
            sys.executable,
            "-m", "uvicorn",
            "app.api:app",
            "--host", "127.0.0.1",
            "--port", "8002"
        ],
        cwd=str(project),
        stdout=log,
        stderr=subprocess.STDOUT
    )

time.sleep(4)

print("Server running:", server.poll() is None)
print("\nServer logs:")
print(log_file.read_text())

Server running: True

Server logs:



In [10]:
import requests

response = requests.post(
    "http://127.0.0.1:8002/predict",
    json={
        "usage": 5,
        "tenure": 8,
        "tickets": 3
    },
    timeout=10
)

print("Status Code:", response.status_code)
print("Prediction:", response.json())

print("\nApplication logs:")
print(log_file.read_text())

Status Code: 200
Prediction: {'churn_probability': 0.7995, 'risk_level': 'High Churn Risk'}

Application logs:
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:     Started server process [5891]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8002 (Press CTRL+C to quit)
INFO:churn_api:method=POST path=/predict status=200 latency_ms=86.12
INFO:     127.0.0.1:49470 - "POST /predict HTTP/1.1" 200 OK



In [11]:
from pathlib import Path

api_file = Path("/content/churn-repo/customer-churn-app/app/api.py")

code = api_file.read_text()

health_endpoint = '''

@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "service": "customer-churn-api"
    }
'''

if '@app.get("/health")' not in code:
    code += health_endpoint
    api_file.write_text(code)

print("Health endpoint added:", '@app.get("/health")' in api_file.read_text())

Health endpoint added: True


In [12]:
import subprocess
import sys
import time

server.terminate()
server.wait(timeout=10)

with open(log_file, "w") as log:
    server = subprocess.Popen(
        [
            sys.executable,
            "-m", "uvicorn",
            "app.api:app",
            "--host", "127.0.0.1",
            "--port", "8002"
        ],
        cwd=str(project),
        stdout=log,
        stderr=subprocess.STDOUT
    )

time.sleep(4)

print("Server running:", server.poll() is None)

Server running: True


In [13]:
import requests

response = requests.get(
    "http://127.0.0.1:8002/health",
    timeout=10
)

print("Status Code:", response.status_code)
print("Response:", response.json())

print("\nApplication logs:")
print(log_file.read_text())

Status Code: 200
Response: {'status': 'healthy', 'service': 'customer-churn-api'}

Application logs:
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:     Started server process [6295]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8002 (Press CTRL+C to quit)
INFO:churn_api:method=GET path=/health status=200 latency_ms=1.44
INFO:     127.0.0.1:42766 - "GET /health HTTP/1.1" 200 OK



In [14]:
from pathlib import Path

api_file = Path("/content/churn-repo/customer-churn-app/app/api.py")
code = api_file.read_text()

if "from fastapi.responses import JSONResponse" not in code:
    code = "from fastapi.responses import JSONResponse\n" + code

handler = '''

@app.exception_handler(Exception)
async def handle_unexpected_error(request: Request, exc: Exception):
    logger.error(
        "Unhandled exception at %s",
        request.url.path,
        exc_info=(type(exc), exc, exc.__traceback__)
    )

    return JSONResponse(
        status_code=500,
        content={"detail": "Internal server error"}
    )
'''

if '@app.exception_handler(Exception)' not in code:
    code += handler

api_file.write_text(code)

print("Exception handler added:",
      '@app.exception_handler(Exception)' in api_file.read_text())

Exception handler added: True


In [15]:
import py_compile

py_compile.compile(
    str(api_file),
    doraise=True
)

print("API syntax validation passed!")

API syntax validation passed!


In [16]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")
tests_dir = project / "tests"
tests_dir.mkdir(exist_ok=True)

test_code = '''
from fastapi.testclient import TestClient
from app.api import app

client = TestClient(app, raise_server_exceptions=False)


def test_health_endpoint():
    response = client.get("/health")

    assert response.status_code == 200
    assert response.json()["status"] == "healthy"


def test_invalid_input():
    response = client.post(
        "/predict",
        json={
            "usage": -5,
            "tenure": 8,
            "tickets": 3
        }
    )

    assert response.status_code == 422


def test_unexpected_error(monkeypatch):
    from app.api import predictor

    def broken_predict(**kwargs):
        raise RuntimeError("Internal test failure")

    monkeypatch.setattr(predictor, "predict", broken_predict)

    response = client.post(
        "/predict",
        json={
            "usage": 5,
            "tenure": 8,
            "tickets": 3
        }
    )

    assert response.status_code == 500
    assert response.json() == {
        "detail": "Internal server error"
    }
    assert "Internal test failure" not in response.text
'''

test_file = tests_dir / "test_reliability.py"
test_file.write_text(test_code)

print("Created:", test_file)

Created: /content/churn-repo/customer-churn-app/tests/test_reliability.py


In [17]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_reliability.py",
        "-v"
    ],
    cwd=str(project),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, langsmith-0.14.2, typeguard-4.6.0
collecting ... collected 3 items

tests/test_reliability.py::test_health_endpoint PASSED                   [ 33%]
tests/test_reliability.py::test_invalid_input PASSED                     [ 66%]
tests/test_reliability.py::test_unexpected_error PASSED                  [100%]

============================== 3 passed in 3.08s ===============================




In [18]:
import subprocess
import sys
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")

result = subprocess.run(
    [
        sys.executable,
        "-m", "pytest",
        "tests/",
        "-v"
    ],
    cwd=str(project),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, langsmith-0.14.2, typeguard-4.6.0
collecting ... collected 13 items / 1 error

==================================== ERRORS ====================================
______________________ ERROR collecting tests/test_api.py ______________________
ImportError while importing test module '/content/churn-repo/customer-churn-app/tests/test_api.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_api.py:4: in <module>
    from api import app
E   ModuleNotFoundError: No module named 'api'
=======

In [19]:
from pathlib import Path

test_file = Path(
    "/content/churn-repo/customer-churn-app/tests/test_api.py"
)

code = test_file.read_text()

code = code.replace(
    "from api import app",
    "from app.api import app"
)

test_file.write_text(code)

print("Import fixed:", "from app.api import app" in test_file.read_text())

Import fixed: True


In [20]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v"],
    cwd="/content/churn-repo/customer-churn-app",
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, langsmith-0.14.2, typeguard-4.6.0
collecting ... collected 19 items

tests/test_api.py::test_home_endpoint PASSED                             [  5%]
tests/test_api.py::test_valid_prediction PASSED                          [ 10%]
tests/test_api.py::test_invalid_requests[customer0] PASSED               [ 15%]
tests/test_api.py::test_invalid_requests[customer1] PASSED               [ 21%]
tests/test_api.py::test_invalid_requests[customer2] PASSED               [ 26%]
tests/test_api.py::test_invalid_requests[customer3] PASSED               [ 31%]
tests/test_prediction.py::test_prediction_probability PASSED             [ 36%]
tests/test_prediction.py::test_customer_risk[5-8-3-High Churn Risk] PASSED [ 42%]
tests/test_

In [21]:
import subprocess

repo = "/content/churn-repo"

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    capture_output=True,
    text=True
)

print(result.stdout)

 M customer-churn-app/app/api.py
 M customer-churn-app/tests/test_api.py
?? customer-churn-app/app/__pycache__/
?? customer-churn-app/tests/__pycache__/
?? customer-churn-app/tests/test_reliability.py
?? customer-churn-app/uvicorn_day12.log



In [22]:
import subprocess

repo = "/content/churn-repo"

commands = [
    ["git", "add",
     "customer-churn-app/app/api.py",
     "customer-churn-app/tests/test_api.py",
     "customer-churn-app/tests/test_reliability.py"],
    ["git", "commit", "-m", "Day 12: Add API logging, health checks and reliability tests"]
]

for command in commands:
    result = subprocess.run(
        command,
        cwd=repo,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        break




Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@785319f7cc30.(none)')



In [23]:
import subprocess

repo = "/content/churn-repo"

subprocess.run(
    ["git", "config", "user.name", "njalagam23"],
    cwd=repo,
    check=True
)

subprocess.run(
    ["git", "config", "user.email", "narasimharaojalagam5@gmail.com"],
    cwd=repo,
    check=True
)

print("Git identity configured successfully!")

Git identity configured successfully!


In [24]:
result = subprocess.run(
    [
        "git", "commit",
        "-m", "Day 12: Add API logging, health checks and reliability tests"
    ],
    cwd=repo,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

[main 7e74b6c] Day 12: Add API logging, health checks and reliability tests
 3 files changed, 101 insertions(+), 1 deletion(-)
 create mode 100644 customer-churn-app/tests/test_reliability.py


Exit code: 0


In [25]:
import subprocess

repo = "/content/churn-repo"

result = subprocess.run(
    ["git", "push", "origin", "HEAD"],
    cwd=repo,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)


fatal: could not read Username for 'https://github.com': No such device or address

Exit code: 128


In [27]:
import getpass
import subprocess
import os

repo = "/content/churn-repo"

username = input("GitHub username: ")
token = getpass.getpass("GitHub personal access token: ")

# Temporary authentication helper for this push only
import tempfile

with tempfile.TemporaryDirectory() as temp_dir:
    askpass_path = os.path.join(temp_dir, "askpass.sh")

    with open(askpass_path, "w") as f:
        f.write(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "$GIT_USERNAME" ;;\n'
            '  *Password*) printf "%s\\n" "$GIT_TOKEN" ;;\n'
            'esac\n'
        )

    os.chmod(askpass_path, 0o700)

    env = os.environ.copy()
    env["GIT_ASKPASS"] = askpass_path
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["GIT_USERNAME"] = username
    env["GIT_TOKEN"] = token

    result = subprocess.run(
        ["git", "push", "origin", "HEAD"],
        cwd=repo,
        env=env,
        capture_output=True,
        text=True
    )

    print(result.stdout)
    print(result.stderr)
    print("Exit code:", result.returncode)

del token

GitHub username: njalagam23
GitHub personal access token: ··········

To https://github.com/njalagam23/neural-networks-from-scratch.git
   d45ba6a..7e74b6c  HEAD -> main

Exit code: 0


In [28]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")

print("Project exists:", project.exists())

if project.exists():
    for item in sorted(project.iterdir()):
        print("📁" if item.is_dir() else "📄", item.name)

    print("\nRequirements:")
    requirements = project / "requirements.txt"
    if requirements.exists():
        print(requirements.read_text())

Project exists: True
📁 .pytest_cache
📁 app
📁 artifacts
📄 main.py
📄 requirements.txt
📁 tests
📄 uvicorn_day12.log

Requirements:
torch
numpy
scikit-learn
joblib



In [29]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")
requirements = project / "requirements.txt"

packages = [
    "torch",
    "numpy",
    "scikit-learn",
    "joblib",
    "fastapi",
    "uvicorn",
    "httpx",
    "pytest"
]

requirements.write_text("\n".join(packages) + "\n")

print(requirements.read_text())

torch
numpy
scikit-learn
joblib
fastapi
uvicorn
httpx
pytest



In [30]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-m", "pip",
        "check"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

ipython 7.34.0 requires jedi, which is not installed.


Exit code: 1
